# Feature Engineering for Forecasting Model

This notebook prepares the cleaned delivery dataset for machine learning. 
The purpose is to transform operational raw variables into mod l-ready features that can help predict delivery service tim age.

## Load cleaned dataset

The cleaned dataset from the previous notebook is loaded as the starting point for feature engineering. The date column is converted to datetime format so that time based variables as year, month and week can be extracted later.

In [2]:
import pandas as pd
import numpy as np

data = pd.read_csv("merged_clean_7years.csv")
data["Dato"] = pd.to_datetime(data["Dato"], errors="coerce")

print("Rows:", len(data))
print("Columns:", len(data.columns))

C:\Users\tuvap\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\tuvap\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (
C:\Users\tuvap\AppData\Local\Temp\ipykernel_15052\4073943458.py:4: DtypeWarning: Columns (0: Rutenr) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv("merged_clean_7years.csv")


Rows: 653213
Columns: 56


## Keep weekday deliveries only

Only Monday to Friday observations are still kept. This is consistent with the operational focus of the thesis, since weekday deliveries represent the regular planning environment.

In [3]:
weekDays = ["Mandag", "Tirsdag", "Onsdag", "Torsdag", "Fredag"]
data = data[data["Ukedag"].isin(weekDays)].copy()

print("Weekday filter after:", len(data))

Rows after weekday filter: 653213


## Create calendar features

Several time related variables are extracted from the delivery date: year, month, ISO week number, day of week. These features help the model capture seasonal patterns, monthly variation and weekday effects.

In [4]:
data["year"] = data["Dato"].dt.year
data["month"] = data["Dato"].dt.month
data["week"] = data["Dato"].dt.isocalendar().week.astype(int)
data["dayOfWeek"] = data["Dato"].dt.weekday

## Define Main Variables

The main column names are stored in variables for easier reuse. This is a standard step that makes the notebook easier to read and work with. It also reduces the risk of typing mistakes later in the analysis.

In [5]:
customerColumn = "Kundenr"
dateColumn = "Dato"
targetColumn = "Leveringstid (min)"

plannedColumn = "Planlagt leveringstid (min)"
deliveryColumn = "Antall leveranser"
weightColumn = "Vekt (kg)"

## Feature Groups

The variables are separated into numerical features, lag features, rolling features, and categorical features. Lag features use previous delivery observations, while rolling features summarise recent patterns such as average level and variation. These types of historical features are commonly used in forecasting and were included to help the model capture recent customer patterns.

In [6]:
numberColumns = ["month", "week", plannedColumn, deliveryColumn, weightColumn]
lagColumns = ["lag_1", "lag_7", "rolling_mean_7", "rolling_std_7"]
categoryColumns = [ "dayOfWeek", "Fylke", "Drivstofftype"]

## Prepare Target Variable

The target variable, delivery service time, is converted to numeric format. Rows with missing target values are removed, since delivery time must be known when training the model.

In [7]:
data[targetColumn] = pd.to_numeric(data[targetColumn], errors="coerce")
rowsBefore = len(data)
data = data.dropna(subset=[targetColumn]).copy()

print("Removed rows:", rowsBefore - len(data))

Removed rows: 0


## Clean numerical features

Selected numerical columns are converted to numeric format. Missing values are replaced with the median value of each column. Median imputation is used because it handles skewed data and extreme values better, which are common in logistics data. Mean imputation was not used because unusual values could affect the average.

In [8]:
for column in numberColumns:
    if column in data.columns:
        data[column] = pd.to_numeric(data[column], errors="coerce")

        medianValue = data[column].median()
        data[column] = data[column].fillna(medianValue)

## Sort data chronologically

The dataset is sorted by customer and date. This is necessary before creating lag features, because historical values must appear in the correct time order.

In [9]:
data = data.sort_values([customerColumn, dateColumn]).reset_index(drop=True)

## Remove duplicate customer-date rows

Duplicate observations for the same customer and date are removed. These rows may occur because of repeated registrations or multiple operational entries. Keeping duplicates could affect the historical features, and create repeated observations in the data. Only the first row is kept.

In [10]:
duplicatesBefore = data.duplicated([customerColumn, dateColumn]).sum()
data = data.drop_duplicates([customerColumn, dateColumn], keep="first").copy()
duplicatesAfter = data.duplicated([customerColumn, dateColumn]).sum()

print("Duplicates before:", duplicatesBefore)
print("Duplicates after:", duplicatesAfter)

Duplicates before: 21041
Duplicates after: 0


## Create Lag Features

Historical service time features are created from previous customer deliveries. lag_1 uses the service time from the previous delivery. lag_7 uses the service time from seven deliveries earlier. These features are added at this stage because the data has been cleaned, sorted by customer and date, and is ready for this type of feature creation. Lag features help the model use recent customer history when predicting the next delivery time. This can improve predictions if delivery times follows similar patterns over time.

In [11]:
data["lag_1"] = data.groupby(customerColumn)[targetColumn].shift(1)
data["lag_7"] = data.groupby(customerColumn)[targetColumn].shift(7)

## Create Rolling Average

A rolling mean over the previous seven observations is calculated for each customer. Only earlier deliveries are used, so no future information is included. This feature gives an average of recent service times, and also creates a smoother view of customer behaviour. It is less affected by single unusual deliveries than one individual observation. The rolling average may help the model identify short term trends in recent delivery patterns, which is key here.

In [12]:
rollingMean = data.groupby(customerColumn)[targetColumn].shift(1)
rollingMean = rollingMean.rolling(7).mean()
rollingMean = rollingMean.reset_index(level=0, drop=True)

data["rolling_mean_7"] = rollingMean

## Create Rolling Variability Feature

After creating the rolling average, a rolling standard deviation over the previous seven observations is also calculated for each customer. Only earlier deliveries are used. While the rolling mean shows the recent average level, this feature shows how stable or unstable recent delivery times have been. Higher values mean larger variation between recent deliveries. This can be useful because customers with more irregular historical patterns may be harder to predict than customers with more consistent behaviour.

In [13]:
rollingStd = data.groupby(customerColumn)[targetColumn].shift(1)
rollingStd = rollingStd.rolling(7).std()
rollingStd = rollingStd.reset_index(level=0, drop=True)

data["rolling_std_7"] = rollingStd

## Remove Rows Without Historical Context

Rows with missing lag or rolling features are removed. These rows are usually found at the start of each customer history, where not enough earlier observations are available. Removing them it ensures that the training data contains complete historical features, and more consistent model input.

In [14]:
data = data.dropna(subset=lagColumns).copy()
print("lag features after:", len(data))

Rows after lag features: 579105


## Create Sequential Time Index

A running counter is created for each customer based on delivery order. This way it gives each customer history a clear sequence structure. The Temporal Fusion Transformer model needs a time index to understand which observations come earlier and later in the series. This step was included because the dataset contains repeated deliveries per customer that may be confusing.

In [15]:
data["time_idx"] = data.groupby(customerColumn).cumcount()

print("Customers:", data[customerColumn].nunique())
print("Max time_idx:", data["time_idx"].max())

Customers: 1978
Max time_idx: 1731


## Prepare Categorical Variables

Selected columns are converted to categorical format. This is done because these variables represent labels rather than numerical values, such as weekday, region, and fuel type. Converting them helps the model treat them as categories instead of just numbers.

In [16]:
for column in categoryColumns:
    if column in data.columns:
        data[column] = data[column].astype(str).astype("category")

## Select Final Modelling Columns

Only the variables needed for modelling are kept in the final dataset. This includes customer ID, date, time index, target variable, numerical features, historical lag and rolling features, and categorical variables. Removing unused columns reduces unnecessary noise, makes the dataset cleaner, and ensures that the model only receives the selected inputs prepared for training.

In [17]:
keepColumns = [customerColumn, dateColumn, "time_idx", targetColumn]
keepColumns += numberColumns
keepColumns += lagColumns

for column in categoryColumns:
    if column in data.columns and column not in keepColumns:
        keepColumns.append(column)
        
modelData = data[keepColumns].copy()

print("Rows:", len(modelData))
print("Columns:", modelData.columns.tolist())

Rows: 579105
Columns: ['Kundenr', 'Dato', 'time_idx', 'Leveringstid (min)', 'month', 'week', 'Planlagt leveringstid (min)', 'Antall leveranser', 'Vekt (kg)', 'lag_1', 'lag_7', 'rolling_mean_7', 'rolling_std_7', 'dayOfWeek', 'Fylke', 'Drivstofftype']


## Save modelling dataset

The final dataset is exported as model_dataset.csv. This file is later used directly for training the forecasting model.

In [18]:
modelData.to_csv("model_dataset.csv", index=False, encoding="utf-8-sig")
print("Saved model_dataset.csv")

Saved model_dataset.csv


## Create demo dataset for prototype

A smaller set of selected dates is saved separately. This dataset is used in the Streamlit prototype to show scenario planning without the full dataset.

In [19]:
demoDates = ["2025-04-22", "2025-06-11", "2025-12-16", "2025-04-25", "2025-06-13"]

In [20]:
demoData = pd.read_csv("merged_clean_7years.csv")
demoData = demoData[demoData["Dato"].astype(str).isin(demoDates)]

print("Rows in demo data:", len(demoData))
print("Dates:", demoData["Dato"].unique())

C:\Users\tuvap\AppData\Local\Temp\ipykernel_15052\1701648943.py:1: DtypeWarning: Columns (0: Rutenr) have mixed types. Specify dtype option on import or set low_memory=False.
  demoData = pd.read_csv("merged_clean_7years.csv")


Rows in demo data: 2078
Dates: <ArrowStringArray>
['2025-04-22', '2025-04-25', '2025-06-11', '2025-06-13', '2025-12-16']
Length: 5, dtype: str


In [44]:
demoData.to_csv("demo_data.csv", index=False)
print("Saved demo_data.csv")

Saved demo_data.csv
